# Task 1 — Data Acquisition & Exploratory Data Analysis
**Heart Disease UCI Dataset**  
MLOps Assignment 01 | AIMLCZG523

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from data_processing import load_raw_data, clean_data, save_processed, COLUMN_NAMES

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../screenshots', exist_ok=True)

## 1.1 Load Raw Data

In [ ]:
df_raw = load_raw_data('../data/raw/processed.cleveland.data')
print('Shape:', df_raw.shape)
df_raw.head(10)

## 1.2 Feature Descriptions

In [ ]:
feature_info = pd.DataFrame({
    'Feature': COLUMN_NAMES,
    'Description': [
        'Age in years', 'Sex (1=male, 0=female)',
        'Chest pain type (1-4)', 'Resting blood pressure (mm Hg)',
        'Serum cholesterol (mg/dl)', 'Fasting blood sugar > 120 mg/dl',
        'Resting ECG results (0-2)', 'Max heart rate achieved',
        'Exercise-induced angina', 'ST depression (oldpeak)',
        'Slope of peak exercise ST', 'Number of major vessels (0-3)',
        'Thal (3=normal, 6=fixed, 7=reversible)', 'Heart disease target (binary)'
    ],
    'Type': ['Continuous','Categorical','Categorical','Continuous','Continuous',
             'Categorical','Categorical','Continuous','Categorical','Continuous',
             'Categorical','Categorical','Categorical','Target']
})
feature_info

## 1.3 Missing Values & Data Quality

In [ ]:
print('Dtypes:\n', df_raw.dtypes)
print('\nMissing values (including ? encoded as NaN):')
print(df_raw.isnull().sum())
print('\nTotal rows with any missing:', df_raw.isnull().any(axis=1).sum())

In [ ]:
# Visualise missing values
fig, ax = plt.subplots(figsize=(10, 3))
missing_pct = (df_raw.isnull().sum() / len(df_raw)) * 100
missing_pct[missing_pct > 0].plot(kind='bar', ax=ax, color='#e74c3c')
ax.set_title('Missing Value Percentage per Feature', fontweight='bold')
ax.set_ylabel('% Missing')
ax.set_xlabel('Feature')
plt.tight_layout()
plt.savefig('../screenshots/eda_missing_values.png')
plt.show()

## 1.4 Clean & Preprocess

In [ ]:
df = clean_data(df_raw)
save_processed(df)
print('Cleaned shape:', df.shape)
df.describe().round(2)

## 1.5 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['target'].value_counts()
labels = ['No Disease (0)', 'Disease (1)']
colors = ['#2ecc71', '#e74c3c']

axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution (Proportion)', fontweight='bold')

plt.tight_layout()
plt.savefig('../screenshots/eda_class_balance.png')
plt.show()
print('Class balance:\n', df['target'].value_counts(normalize=True).round(3))

## 1.6 Feature Distributions

In [ ]:
continuous = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(continuous):
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        axes[i].hist(df[df['target'] == label][col], bins=20, alpha=0.6,
                     color=color, label=f'Target={label}', edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=9)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

axes[-1].axis('off')
fig.suptitle('Continuous Feature Distributions by Target Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/eda_continuous_distributions.png')
plt.show()

## 1.7 Categorical Feature Counts

In [ ]:
categorical = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(categorical):
    ct = df.groupby([col, 'target']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'],
            edgecolor='white', linewidth=1)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(['No Disease', 'Disease'], fontsize=8)

fig.suptitle('Categorical Feature Counts by Target Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/eda_categorical_features.png')
plt.show()

## 1.8 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../screenshots/eda_correlation_heatmap.png')
plt.show()

## 1.9 Boxplots: Continuous Features vs Target

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for i, col in enumerate(continuous):
    df.boxplot(column=col, by='target', ax=axes[i],
               boxprops=dict(color='#2c3e50'),
               medianprops=dict(color='#e74c3c', linewidth=2))
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Target (0=No Disease, 1=Disease)')

plt.suptitle('Continuous Features by Target Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/eda_boxplots.png')
plt.show()

## 1.10 EDA Summary

In [ ]:
print('='*55)
print('EDA Summary')
print('='*55)
print(f'Total samples          : {len(df)}')
print(f'Features               : {len(df.columns)-1}')
print(f'Missing values (after) : {df.isnull().sum().sum()}')
print(f'Class 0 (No Disease)   : {(df.target==0).sum()} ({(df.target==0).mean()*100:.1f}%)')
print(f'Class 1 (Disease)      : {(df.target==1).sum()} ({(df.target==1).mean()*100:.1f}%)')
print('\nTop correlations with target:')
print(df.corr()['target'].abs().sort_values(ascending=False)[1:6])